# minBPE
一个tokenizer，运用byte pair encoding的方法，也就是两个两个字符地检查，最后把出现次数最多的那个字符对替换掉，整体变成一个新的字符串，然后循环多次  

The Tokenizer is a completely separate, independent module from the LLM. It has its own training dataset of text (which could be different from that of the LLM), on which you train the vocabulary using the Byte Pair Encoding (BPE) algorithm. It then translates back and forth between raw text and sequences of tokens. The LLM later only ever sees the tokens and never directly deals with any text.
<img src="tokenizer.png" style="width:70%">

#### 转换成byte，计算次数

In [ ]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # 一个 bytes 对象，里面包含了 text 的 UTF-8 编码结果
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience。map(int, tokens) 是一个函数调用，它将 tokens 中的每个元素（即每个字节）转换为一个整数。list() 函数将 map 对象转换为一个列表，从而得到一个包含所有字节整数的列表。这使得我们可以更方便地查看和处理文本的 UTF-8 编码结果。每个整数代表 UTF-8 编码中的一个字节，范围在 0 到 255 之间。

print(text)
print("length:", len(text))
print(tokens)
print("length:", len(tokens))

Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 1

In [ ]:
def get_stats(ids): # 统计连续字节对的出现次数，返回一个字典，其中键是字节对（一个元组），值是该字节对出现的次数。
    counts = {}
    for pair in zip(ids, ids[1:]): # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1 # counts.get(pair, 0) 返回 counts 字典中键 pair 的值，如果 pair 不存在，则返回默认值 0
    return counts

stats = get_stats(tokens)
print(stats)
print(sorted(((v,k) for k,v in stats.items()), reverse=True)) # sorted() 函数用于对可迭代对象进行排序。这里的 ((v,k) for k,v in stats.items()) 是一个生成器表达式，它将 stats 字典中的每个键值对 (k, v) 转换为一个元组 (v, k)，其中 v 是出现次数，k 是字节对。sorted() 函数会根据元组的第一个元素（即出现次数v）进行排序，reverse=True 参数表示降序排序。最终输出的是一个列表，其中每个元素都是一个元组，包含出现次数和对应的字节对，按照出现次数从高到低排序。

{(239, 188): 1, (188, 181): 1, (181, 239): 1, (239, 189): 6, (189, 142): 1, (142, 239): 1, (189, 137): 1, (137, 239): 1, (189, 131): 1, (131, 239): 1, (189, 143): 1, (143, 239): 1, (189, 132): 1, (132, 239): 1, (189, 133): 1, (133, 33): 1, (33, 32): 2, (32, 240): 3, (240, 159): 15, (159, 133): 7, (133, 164): 1, (164, 240): 1, (133, 157): 1, (157, 240): 1, (133, 152): 1, (152, 240): 1, (133, 146): 1, (146, 240): 1, (133, 158): 1, (158, 240): 1, (133, 147): 1, (147, 240): 1, (133, 148): 1, (148, 226): 1, (226, 128): 12, (128, 189): 1, (189, 32): 1, (159, 135): 7, (135, 186): 1, (186, 226): 1, (128, 140): 6, (140, 240): 6, (135, 179): 1, (179, 226): 1, (135, 174): 1, (174, 226): 1, (135, 168): 1, (168, 226): 1, (135, 180): 1, (180, 226): 1, (135, 169): 1, (169, 226): 1, (135, 170): 1, (170, 33): 1, (159, 152): 1, (152, 132): 1, (132, 32): 1, (32, 84): 1, (84, 104): 1, (104, 101): 6, (101, 32): 20, (32, 118): 1, (118, 101): 3, (101, 114): 6, (114, 121): 2, (121, 32): 2, (32, 110): 2, (110,

#### 替换字节

In [ ]:
top_pair = max(stats, key=stats.get) # key=stats.get 表示在比较字典的键时，使用对应的值（即出现次数）进行比较。最终返回的是出现次数最多的字节对（即 top_pair）。

def merge(ids, pair, idx):
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]: # 这里的条件判断确保我们不会越界访问列表（第一个条件检查的），并且检查当前元素和下一个元素是否匹配指定的字节对（后两个条件检查的）。如果匹配成功，我们将这个字节对替换为新的索引（idx）。
      newids.append(idx) # 如果条件满足，我们将新的索引 idx （将这个字节对替换为一个新的单一标记）添加到 newids 列表中
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

# 一个例子
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99)) 

# 对tokens进行操作
tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))

' '

#### 完整实现

In [9]:
# making the training text longer to have more representative token statistics
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
# 读取文本数据
with open("text_for_minBPE.txt", "r", encoding="utf-8") as f:
    text = f.read()
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx): 
  ''' 在 ids 中将所有连续出现的 pair 替换为新的索引 idx '''
  newids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

# ---
vocab_size = 276 # the desired final vocabulary size。因为每次替换一个字节对会引入一个新的标记，所以我们需要进行 vocab_size - 256 次替换（因为初始的标记是 0-255 的单字节标记）。
num_merges = vocab_size - 256 # 替换字节对的次数
ids = list(tokens) # 创建一个tokens的复制，so we don't destroy the original list

merges = {} # (int, int) -> int ，记录每次替换的字节对和对应的新索引
# 进行 vocab_size - 256 次替换：
for i in range(num_merges): 
  stats = get_stats(ids) # 统计当前 ids 中连续字节对的出现次数，返回一个字典，其中键是字节对（一个元组），值是该字节对出现的次数。
  pair = max(stats, key=stats.get) # 从统计结果中找到出现次数最多的字节对（即 pair）
  idx = 256 + i # new token id
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx) # 在 ids 中将所有连续出现的 pair 替换为新的索引 idx
  merges[pair] = idx # 记录这次替换的字节对和对应的新索引

merging (101, 32) into a new token 256
merging (105, 110) into a new token 257
merging (115, 32) into a new token 258
merging (116, 104) into a new token 259
merging (101, 114) into a new token 260
merging (99, 111) into a new token 261
merging (116, 32) into a new token 262
merging (226, 128) into a new token 263
merging (44, 32) into a new token 264
merging (97, 110) into a new token 265
merging (111, 114) into a new token 266
merging (100, 32) into a new token 267
merging (97, 114) into a new token 268
merging (101, 110) into a new token 269
merging (257, 103) into a new token 270
merging (261, 100) into a new token 271
merging (121, 32) into a new token 272
merging (46, 32) into a new token 273
merging (97, 108) into a new token 274
merging (259, 256) into a new token 275


In [11]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X") # 压缩比

tokens length: 24636
ids length: 19484
compression ratio: 1.26X


#### decoding和encoding
decoding：把token变成原始字符  
encoding：把原始字符变成token

In [ ]:
# decoding

vocab = {idx: bytes([idx]) for idx in range(256)} # vocab 字典，将每个数（0-255）映射到对应的字节值（bytes([idx])）
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1] # 对于每个被替换的字节对 (p0, p1) 和对应的新索引 idx，vocab[idx] 为 vocab[p0] 和 vocab[p1] 的连接。按merges中替换的顺序，最后所有的新标记都会还原成原始的字节序列。

def decode(ids):
  # given ids (list of integers), return Python string
  tokens = b"".join(vocab[idx] for idx in ids) # 对于输入的 ids 列表中的每个索引 idx，vocab[idx] 返回对应的字节序列。b"".join(...) 将这些字节序列连接成一个完整的字节串。vocab[idx] for idx in ids 是一个生成器表达式
  text = tokens.decode("utf-8", errors="replace") # 将连接后的字节串解码为一个 Python 字符串，使用 UTF-8 编码。如果在解码过程中遇到无效的字节序列，errors="replace" 参数会将这些无效的字节替换为一个特殊的替代字符（通常是 �），以避免抛出异常。
  return text

print(decode([128])) # 128会返回异常，和utf编码规则有关

�


In [36]:
# encoding

def encode(text):
  # given a string, return list of integers (the tokens)
  tokens = list(text.encode("utf-8")) # 将输入的文本字符串编码为 UTF-8 字节串，并将其转换为一个整数列表
  while len(tokens) >= 2: # 只要 tokens 中至少有两个元素，我们就继续进行替换。因为替换的对象是连续的字节对
    stats = get_stats(tokens) # 统计当前 tokens 中连续字节对的出现次数，返回一个字典
    pair = min(stats, key=lambda p: merges.get(p, float("inf"))) # key=lambda p: merges.get(p, float("inf")) 表示看字典的每个键p时，使用对应的值（即出现次数）进行比较。如果某个字节对p不在 merges 字典中，get 方法会返回 float("inf")，这意味着这个字节对不会被选择为最小的，因为它的出现次数被视为无穷大，否则返回在 merges 字典中的编号。最后返回值最小的键
    if pair not in merges: # 如果当前最小的字节对不在 merges 字典中，说明没有更多的字节对可以替换了，我们就停止替换过程。
      break # nothing else can be merged
    idx = merges[pair]
    tokens = merge(tokens, pair, idx) 
  return tokens

print(encode("fsbointebptewi"))

[102, 115, 98, 111, 257, 116, 101, 98, 112, 116, 101, 119, 105]


In [37]:
# 测试
valtext = "Many common characters, including numerals, punctuation, and other symbols, are unified within the standard and are not treated as specific to any given writing system. Unicode encodes thousands of emoji, with the continued development thereof conducted by the Consortium as a part of the standard.[4] Moreover, the widespread adoption of Unicode was in large part responsible for the initial popularization of emoji outside of Japan. Unicode is ultimately capable of encoding more than 1.1 million characters."
valtext2 = decode(encode(valtext))
print(valtext2 == valtext)

True


#### Forced splits using regex patterns (GPT series)
有一些字符需要被强制分开

In [ ]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""") # 这里的正则表达式模式 是一个复杂的模式，用于匹配文本中的不同类型的标记

print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']


#### GPT的tokenizer

In [41]:
# pip install tiktoken
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


In [ ]:
# wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
# wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

In [ ]:
import os, json

with open('encoder.json', 'r') as f:
    encoder = json.load(f) # <--- equivalent to our "vocab"，也就是一个字典，将每个标记（无论是单字节还是合并后的标记）映射到一个唯一的整数索引。

with open('vocab.bpe', 'r', encoding="utf-8") as f:
    bpe_data = f.read()
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]  #<---- equivalent to our "merges"，也就是一个列表，其中每个元素是一个元组，表示被合并的字节对。每行的格式是 "byte1 byte2"，通过 split() 方法将其分割成两个部分，并转换为一个元组。最终得到的 bpe_merges 列表包含了所有的字节对替换规则，按照它们在 vocab.bpe 文件中的顺序排列。

#### special tokens
special token 是指在编码过程中可能会使用到的一些特殊标记，例如用于表示文本开头、结尾、未知字符等。这些特殊标记通常会被分配一个独特的索引，以便在编码和解码过程中进行区分。

In [44]:
len(encoder) # 256 raw byte tokens. 50,000 merges. +1 special token。

50257

In [45]:
encoder['<|endoftext|>'] # the only special token in use for the GPT-2 base model

50256

#### Sentencepiece
另一种tokenizer

#### SolidGoldMagikarp 现象

SolidGoldMagikarp 现象是一个典型的后门攻击现象，当对 ChatGPT 输入 “SolidGoldMagikarp”时，它只回答一个词：“distribute”。当让它重复 “StreamerBot”时，它会回复：“You’re a jerk”。当被要求重复 “TheNitromeFan”时，它的回答是“182”。而如果在这个词两边加上单引号，他的回答就是无穷无尽的“ The”。当被问及 TheNitromeFan 是谁时，ChatGPT 回答说：“182 是一个数字，不是一个人。它通常被用来指代这个数字本身。” 

产生该现象的原因是gpt的tokenizer和模型训练是分开的，soligoldmagikarp是一个在reddit论坛上发了很多帖子的用户，训练tokenizer时专门将他的名字作为一个tokenizer，但是在模型训练时数据集中没有这个名字，因此这一token的embedding没有得到训练，最后输出的结果是随机的。